In [10]:
# -------------------------------
# 1️⃣ Import Libraries
# -------------------------------
import tensorflow as tf
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.layers import Input, Flatten, Dense, Dropout
from tensorflow.keras.applications import VGG16
import numpy as np
import cv2
import os
from google.colab import drive

# -------------------------------
# 2️⃣ Mount Google Drive
# -------------------------------
drive.mount('/content/drive')

# -------------------------------
# 3️⃣ Load Sequential model weights
# -------------------------------
seq_model_path = '/content/drive/MyDrive/vgg16_brain_tumor_best.keras'
seq_model = load_model(seq_model_path)
print("✅ Sequential model loaded")

# -------------------------------
# 4️⃣ Rebuild Functional Model
# -------------------------------
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224,224,3))
for layer in base_model.layers[:-4]:
    layer.trainable = False

x = Flatten()(base_model.output)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(4, activation='softmax')(x)

func_model = Model(inputs=base_model.input, outputs=output)
func_model.set_weights(seq_model.get_weights())  # Load the trained weights
print("✅ Functional model rebuilt and weights loaded")

# -------------------------------
# 5️⃣ Find last conv layer
# -------------------------------
for layer in reversed(base_model.layers):
    if 'conv' in layer.name:
        last_conv_layer = layer
        break
print("✅ Last conv layer:", last_conv_layer.name)

# -------------------------------
# 6️⃣ Grad-CAM function
# -------------------------------
def make_gradcam_heatmap(img_tensor, model, last_conv_layer):
    grad_model = Model(inputs=model.input,
                       outputs=[last_conv_layer.output, model.output])
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_tensor)
        pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]
    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0,1,2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy(), pred_index.numpy()

# -------------------------------
# 7️⃣ Overlay heatmap function
# -------------------------------
def overlay_heatmap(img_path, heatmap, alpha=0.4, colormap=cv2.COLORMAP_JET):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (224,224))
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap = np.uint8(255 * heatmap)
    heatmap_color = cv2.applyColorMap(heatmap, colormap)
    superimposed_img = cv2.addWeighted(img, 1-alpha, heatmap_color, alpha, 0)
    return superimposed_img

# -------------------------------
# 8️⃣ Paths
# -------------------------------
test_dir = '/content/drive/MyDrive/MRI_Images/Testing/'
output_dir = '/content/GradCAM_Results/'
os.makedirs(output_dir, exist_ok=True)
classes = sorted(os.listdir(test_dir))

# -------------------------------
# 9️⃣ Generate Grad-CAM
# -------------------------------
for cls in classes:
    class_path = os.path.join(test_dir, cls)
    save_class_dir = os.path.join(output_dir, cls)
    os.makedirs(save_class_dir, exist_ok=True)

    img_files = [f for f in os.listdir(class_path) if f.endswith(('.jpg','.png','.jpeg'))][:5]

    for img_file in img_files:
        img_path = os.path.join(class_path, img_file)
        img = tf.keras.utils.load_img(img_path, target_size=(224,224))
        img_tensor = tf.keras.utils.img_to_array(img)
        img_tensor = np.expand_dims(img_tensor, axis=0) / 255.0

        heatmap, pred_index = make_gradcam_heatmap(img_tensor, func_model, last_conv_layer)
        superimposed_img = overlay_heatmap(img_path, heatmap)

        save_path = os.path.join(save_class_dir, f'gradcam_{img_file}')
        cv2.imwrite(save_path, superimposed_img)
        print(f"✅ Saved Grad-CAM for {cls}/{img_file} at {save_path}")

print("🎉 Grad-CAM images saved successfully!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Sequential model loaded
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
✅ Functional model rebuilt and weights loaded
✅ Last conv layer: block5_conv3
✅ Saved Grad-CAM for glioma/Te-glTr_0001.jpg at /content/GradCAM_Results/glioma/gradcam_Te-glTr_0001.jpg
✅ Saved Grad-CAM for glioma/Te-glTr_0004.jpg at /content/GradCAM_Results/glioma/gradcam_Te-glTr_0004.jpg
✅ Saved Grad-CAM for glioma/Te-glTr_0003.jpg at /content/GradCAM_Results/glioma/gradcam_Te-glTr_0003.jpg
✅ Saved Grad-CAM for glioma/Te-glTr_0000.jpg at /content/GradCAM_Results/glioma/gradcam_Te-glTr_0000.jpg
✅ Saved Grad-CAM for glioma/Te-glTr_0002.jpg at /content/GradCAM_Results/glioma/gradcam_Te-glTr_0002.jpg
✅ Saved Grad-CAM for meningioma/Te-meTr_0002.jpg at /content/GradCAM_Results/meningioma/gradcam_Te-meTr_0002.jpg
✅ Saved Grad-CAM for meningioma/Te-meTr_0000.jpg at /content/GradCAM_Results/m

In [11]:
from google.colab import files
import shutil

# Zip the GradCAM_Results folder
shutil.make_archive('/content/GradCAM_Results', 'zip', '/content/GradCAM_Results')

# Download the zip
files.download('/content/GradCAM_Results.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>